# Olympic data extraction — one file per visualisation

Input: `olympics.csv` (the merged dataset, 22 columns)

| Output file | Visualisation |
|---|---|
| `gender_evolution.json` | gender-evolution.js |
| `delegations.csv` | delegations.js (Sankey) |
| `never_medaled.csv` | never-medaled.js |
| `body_types.csv` | body-types.js |
| `fair_distribution.csv` | fair-distribution.js |
| `geopolitics.csv` | geopolitics-conflicts.js |

In [15]:
import numpy as np
import pandas as pd
import json, re, os

INPUT     = 'olympics.csv'
OUT_DIR   = '../../data/js'
STOP_YEAR = 2016

os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT)
df.columns = [c.lower() for c in df.columns]
df = df[df['year'] <= STOP_YEAR]

print('Loaded:', df.shape)
print('Columns:', list(df.columns))

Loaded: (271116, 22)
Columns: ['id', 'name', 'gender', 'age', 'height', 'weight', 'team', 'noc', 'country', 'population', 'year', 'season', 'city', 'sport', 'event', 'medal', 'gdp_per_capita', 'conflict_name', 'conflict_start', 'conflict_end', 'conflict_reason', 'conflict_result']


## 1. Gender evolution (`gender_evolution.json`)

Fully pre-aggregated nested JSON: season → year → sport → discipline → {men, women}.
JS only builds Map structures from it — zero computation at runtime.

In [16]:
MEN_VALUES   = {'m', 'men', 'male', 'man'}
WOMEN_VALUES = {'f', 'women', 'female', 'woman'}
STRIP = re.compile(r"\b(women's|womens|men's|mens|women|men|female|male|mixed)\b", re.IGNORECASE)

def norm_gender(v):
    s = str(v).strip().lower()
    if s in MEN_VALUES:   return 'men'
    if s in WOMEN_VALUES: return 'women'
    return None

def norm_discipline(event, sport):
    text = STRIP.sub('', str(event)).strip()
    text = re.compile(r'^' + re.escape(str(sport)) + r'\s+', re.IGNORECASE).sub('', text).strip()
    return re.sub(r'\s+', ' ', text).strip(' -,:').strip() or str(sport)

g = df.copy()
g['_gender']     = g['gender'].apply(norm_gender)
g['_sport']      = g['sport'].str.strip()
g['_discipline'] = g.apply(lambda r: norm_discipline(r['event'], r['_sport']), axis=1)
g = g[g['_gender'].notna()]

agg = (
    g.groupby(['season', 'year', '_sport', '_discipline', '_gender'])
     .size().reset_index(name='count')
)

result = {}
for _, row in agg.iterrows():
    season, year, sport, disc, gender, count = (
        row['season'], str(int(row['year'])), row['_sport'],
        row['_discipline'], row['_gender'], int(row['count'])
    )
    result.setdefault(season, {})
    result[season].setdefault(year, {'totals': {'men': 0, 'women': 0}, 'sports': {}})
    yn = result[season][year]
    yn['totals'][gender] += count
    yn['sports'].setdefault(sport, {'men': 0, 'women': 0, 'disciplines': {}})
    sn = yn['sports'][sport]
    sn[gender] += count
    sn['disciplines'].setdefault(disc, {'men': 0, 'women': 0})
    sn['disciplines'][disc][gender] += count

out = f'{OUT_DIR}/gender_evolution.json'
with open(out, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, separators=(',', ':'))
print(f'Written {out}  ({os.path.getsize(out)/1024:.0f} KB)')

Written ../../data/js/gender_evolution.json  (239 KB)


## 2. Medal Sankey (`delegations.csv`)

One row per unique medal awarded (deduplicated by country+event+medal).
Columns needed by delegations.js: year, season, noc, team, sport, event, medal.

In [17]:
delegations = (
    df[df['medal'].notna() & df['medal'].str.strip().isin(['Gold','Silver','Bronze'])]
    .drop_duplicates(subset=['year','season','noc','team','event','medal'])
    [['year','season','noc','team','sport','event','medal']]
    .reset_index(drop=True)
)
out = f'{OUT_DIR}/delegations.csv'
delegations.to_csv(out, index=False)
print(f'Written {out}  ({len(delegations):,} rows,  {os.path.getsize(out)/1024:.0f} KB)')

Written ../../data/js/delegations.csv  (18,927 rows,  1421 KB)


## 3. Never-medaled (`never_medaled.csv`)

One row per athlete-participation. JS checks cumulative medal history up to year N
and computes participation counts — minimal columns needed.

In [18]:
never = df[['year','season','country','noc','sport','medal']].copy()
out = f'{OUT_DIR}/never_medaled.csv'
never.to_csv(out, index=False)
print(f'Written {out}  ({len(never):,} rows,  {os.path.getsize(out)/1024:.0f} KB)')

Written ../../data/js/never_medaled.csv  (271,116 rows,  9531 KB)


## 4. Body types (`body_types.csv`)

Medal winners only, one row per unique medalist per event.
Discipline name is pre-normalised here so JS does zero string munging.

In [19]:
STRIP2 = re.compile(r"\b(women's|womens|men's|mens|women|men|mixed)\b", re.IGNORECASE)

def norm_disc(event, sport):
    text = STRIP2.sub('', str(event)).strip()
    text = re.compile(r'^' + re.escape(str(sport)) + r'\s*', re.IGNORECASE).sub('', text).strip()
    return re.sub(r'\s+', ' ', text).strip() or str(sport)

body = df[
    df['medal'].notna() & df['medal'].str.strip().isin(['Gold','Silver','Bronze'])
].copy()

body['discipline'] = body.apply(lambda r: norm_disc(r['event'], r['sport']), axis=1)
body['gender'] = body['gender'].apply(
    lambda v: 'Women' if str(v).strip().lower() in ('f','female','women','woman') else 'Men'
)

body = (
    body
    .drop_duplicates(subset=['year','season','name','event','medal'])
    [['year','season','gender','sport','discipline','name','noc','team','height','weight','medal']]
    .reset_index(drop=True)
)
out = f'{OUT_DIR}/body_types.csv'
body.to_csv(out, index=False)
print(f'Written {out}  ({len(body):,} rows,  {os.path.getsize(out)/1024:.0f} KB)')

Written ../../data/js/body_types.csv  (39,768 rows,  3680 KB)


## 5. Fair distribution (`fair_distribution.csv`)

One row per country+year+season. Medal counts (gold/silver/bronze) are already
computed here. JS renders bars and computes the fair factor — no groupby needed.

In [20]:
# ── actual medal counts (deduplicated by country+event+medal) ────────────────
medals_raw = (
    df[df['medal'].notna() & df['medal'].str.strip().isin(['Gold','Silver','Bronze'])]
    .drop_duplicates(subset=['year','season','country','event','medal'])
)

medal_counts = (
    medals_raw
    .groupby(['year','season','country'])
    .apply(lambda g: pd.Series({
        'gold':   (g['medal'] == 'Gold').sum(),
        'silver': (g['medal'] == 'Silver').sum(),
        'bronze': (g['medal'] == 'Bronze').sum(),
    }), include_groups=False)
    .reset_index()
)

# ── one gdp + population per country+year+season ─────────────────────────────
econ = (
    df.groupby(['year','season','country'], as_index=False)
    .agg(gdp_per_capita=('gdp_per_capita','first'),
         population=('population','first'))
)

# ── merge: every participating country gets a row ───────────────────────────
fair = (
    econ
    .merge(medal_counts, on=['year','season','country'], how='left')
    .fillna({'gold': 0, 'silver': 0, 'bronze': 0})
)
fair[['gold','silver','bronze']] = fair[['gold','silver','bronze']].astype(int)

fair = (
    fair[['year','season','country','gdp_per_capita','population',
          'gold','silver','bronze']]
    .reset_index(drop=True)
)

out = f'{OUT_DIR}/fair_distribution.csv'
fair.to_csv(out, index=False)
print(f'Written {out}  ({len(fair):,} rows,  {os.path.getsize(out)/1024:.0f} KB)')

Written ../../data/js/fair_distribution.csv  (3,786 rows,  176 KB)


## 6. Geopolitics (`geopolitics.csv`)

One row per country+year (Summer only). Pre-computed medal count so JS only
needs to plot bars for selected countries — no groupby at runtime.

In [21]:
summer = df[df['season'] == 'Summer'].copy()

geo_medals = (
    summer[
        summer['medal'].notna() &
        summer['medal'].str.strip().isin(['Gold','Silver','Bronze'])
    ]
    .drop_duplicates(subset=['year','country','event','medal'])
    .groupby(['year','country'], as_index=False)
    .size()
    .rename(columns={'size': 'medal_count'})
)

geo_all = (
    summer[['year','country','noc',
            'conflict_name','conflict_start','conflict_end',
            'conflict_reason','conflict_result']]
    .drop_duplicates(subset=['year','country'])
)

geo = (
    geo_all
    .merge(geo_medals, on=['year','country'], how='left')
    .fillna({'medal_count': 0})
)
geo['medal_count'] = geo['medal_count'].astype(int)
# Track whether the country participated (always True here, but useful flag)
geo['participated'] = True

out = f'{OUT_DIR}/geopolitics.csv'
geo.to_csv(out, index=False)
print(f'Written {out}  ({len(geo):,} rows,  {os.path.getsize(out)/1024:.0f} KB)')

Written ../../data/js/geopolitics.csv  (2,786 rows,  88 KB)


## 7. Summary

In [22]:
files = ['gender_evolution.json','delegations.csv','never_medaled.csv',
         'body_types.csv','fair_distribution.csv','geopolitics.csv']
print(f'{"File":<30} {"Size (KB)":>10}')
print('-'*42)
total = 0
for fname in files:
    kb = os.path.getsize(f'{OUT_DIR}/{fname}') / 1024
    total += kb
    print(f'{fname:<30} {kb:>10.0f}')
print('-'*42)
print(f'{"TOTAL":<30} {total:>10.0f}')

File                            Size (KB)
------------------------------------------
gender_evolution.json                 239
delegations.csv                      1421
never_medaled.csv                    9531
body_types.csv                       3680
fair_distribution.csv                 176
geopolitics.csv                        88
------------------------------------------
TOTAL                               15135
